# EyeAI AMD3 — Freeze Run 09 + TTA Model Package

This notebook does **not** train a model. It converts the saved Run 09 champion checkpoint and TTA result into a portable inference package.

Output:

```text
/kaggle/working/eyeai_model_package/run09_tta_v1
```

The package becomes the stable input for the standalone inference engine and the next FastAPI stage.

## 1. Paths and export controls

Attach the saved Notebook output containing Run 09 and Run 09 TTA. Explicit overrides are optional when automatic discovery finds one unique file.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

REPO_OWNER = "MozaicAI-Solutions"
REPO_NAME = "eyeai-team-AMD3"
BRANCH = "main"
REPO_DIR = Path("/kaggle/working/eyeai-team-AMD3")

MODEL_PACKAGE_CONFIG = REPO_DIR / "configs/model_packages/run09_tta_v1.yaml"
MODEL_PACKAGE_DIR = Path("/kaggle/working/eyeai_model_package/run09_tta_v1")

RUN09_CHECKPOINT_OVERRIDE = None
TTA_SUMMARY_OVERRIDE = None
TRAINING_SUMMARY_OVERRIDE = None

RUN_SMOKE_TEST = True
SMOKE_TEST_IMAGE_COUNT = 3

print("Model package output:", MODEL_PACKAGE_DIR)

## 2. Clone the repository and install EyeAI

This pulls the committed package-export and inference files, then installs the local Python package.

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("GITHUB_TOKEN secret is missing.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
print("Repository and package are ready.")

## 3. Discover the saved Run 09 artifacts

The champion checkpoint is required. The TTA and training summaries are copied into the package when available.

In [ ]:
RUN09_FILENAME = "retfound_cfp_run09_last10_mixed_best.pth"
TTA_SUMMARY_FILENAME = "run09_tta_summary.json"
TRAINING_SUMMARY_FILENAME = "retfound_cfp_run09_last10_mixed_summary.json"


def resolve_file(filename: str, override=None, preferred_fragments=(), required=True):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path

    candidates = sorted({
        path
        for root in [Path("/kaggle/input"), Path("/kaggle/working")]
        if root.exists()
        for path in root.rglob(filename)
        if path.is_file()
    })

    for fragment in preferred_fragments:
        preferred = [path for path in candidates if fragment in str(path)]
        if len(preferred) == 1:
            return preferred[0]

    if len(candidates) == 1:
        return candidates[0]
    if not candidates and not required:
        return None
    if not candidates:
        raise FileNotFoundError(
            f"Could not find {filename} under /kaggle/input or /kaggle/working."
        )
    raise RuntimeError(
        f"Multiple candidates found for {filename}. Set its override explicitly:\n"
        + "\n".join(f"- {path}" for path in candidates)
    )


RUN09_CHECKPOINT = resolve_file(
    RUN09_FILENAME,
    override=RUN09_CHECKPOINT_OVERRIDE,
    preferred_fragments=("run09_retfound_last10/checkpoints",),
)
TTA_SUMMARY = resolve_file(
    TTA_SUMMARY_FILENAME,
    override=TTA_SUMMARY_OVERRIDE,
    preferred_fragments=("evaluations/run09_tta", "run09_tta"),
    required=False,
)
TRAINING_SUMMARY = resolve_file(
    TRAINING_SUMMARY_FILENAME,
    override=TRAINING_SUMMARY_OVERRIDE,
    preferred_fragments=("run09_retfound_last10/logs",),
    required=False,
)

print("Run 09 checkpoint:", RUN09_CHECKPOINT)
print("TTA summary:", TTA_SUMMARY)
print("Training summary:", TRAINING_SUMMARY)
print(f"Checkpoint size: {RUN09_CHECKPOINT.stat().st_size / (1024 ** 3):.2f} GB")

## 4. Export the inference-only model package

The exporter keeps the complete trained model state but removes optimizer, AMP scaler, and resume-only training state. No retraining occurs.

In [ ]:
if MODEL_PACKAGE_DIR.exists():
    shutil.rmtree(MODEL_PACKAGE_DIR)

command = [
    "python", "-u", "scripts/export_run09_model_package.py",
    "--checkpoint", str(RUN09_CHECKPOINT),
    "--package-config", str(MODEL_PACKAGE_CONFIG),
    "--output-dir", str(MODEL_PACKAGE_DIR),
    "--training-config", str(REPO_DIR / "configs/train_retfound_binary_run09_last10_mixed.yaml"),
    "--repo-dir", str(REPO_DIR),
]
if TTA_SUMMARY is not None:
    command.extend(["--tta-summary", str(TTA_SUMMARY)])
if TRAINING_SUMMARY is not None:
    command.extend(["--training-summary", str(TRAINING_SUMMARY)])

subprocess.run(command, check=True, cwd=REPO_DIR)
print("Export completed:", MODEL_PACKAGE_DIR)

## 5. Validate the exported package

This confirms all deployment contracts exist and displays provenance, threshold, and metrics before the package is saved.

In [ ]:
required_files = [
    "model.pth",
    "model_config.yaml",
    "preprocessing.json",
    "threshold.json",
    "labels.json",
    "metrics.json",
    "model_card.md",
    "version.json",
    "artifact_manifest.json",
    "README.md",
]

missing = [name for name in required_files if not (MODEL_PACKAGE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Model package is incomplete: {missing}")

print("Package files:")
for path in sorted(MODEL_PACKAGE_DIR.rglob("*")):
    if path.is_file():
        print(f"- {path.relative_to(MODEL_PACKAGE_DIR)}: {path.stat().st_size / (1024 ** 2):.2f} MB")

print("\nThreshold contract:")
print(json.dumps(json.loads((MODEL_PACKAGE_DIR / "threshold.json").read_text()), indent=2))

print("\nVersion and provenance:")
print(json.dumps(json.loads((MODEL_PACKAGE_DIR / "version.json").read_text()), indent=2))

## 6. Standalone smoke test on three images

The package is loaded without the original RETFound checkpoint. Each image is processed from its saved fundus file and evaluated with original + horizontal-flip TTA.

In [ ]:
if RUN_SMOKE_TEST:
    import pandas as pd
    import matplotlib.pyplot as plt
    from PIL import Image

    from eyeai.inference.run09_predictor import Run09Predictor

    dataset_roots = [
        Path("/kaggle/input/datasets/alihasan15/hymd-armd-dataset/eyeai_prepared_binary_dataset"),
        Path("/kaggle/working/eyeai_prepared_binary_dataset_retfound"),
    ]
    dataset_roots.extend(
        path.parent
        for path in Path("/kaggle/input").rglob("dataset_summary.json")
        if (path.parent / "manifests/hyamd_val.csv").is_file()
    )
    DATASET_ROOT = next(
        (root for root in dataset_roots if (root / "manifests/hyamd_val.csv").is_file()),
        None,
    )
    if DATASET_ROOT is None:
        raise FileNotFoundError("Attach the prepared HYAMD + ARMD dataset for the smoke test.")

    validation_df = pd.read_csv(DATASET_ROOT / "manifests/hyamd_val.csv", dtype={"image_id": str})
    examples = validation_df.sample(
        n=min(SMOKE_TEST_IMAGE_COUNT, len(validation_df)),
        random_state=42,
    )

    predictor = Run09Predictor(MODEL_PACKAGE_DIR)
    figure, axes = plt.subplots(len(examples), 1, figsize=(8, 5 * len(examples)))
    if len(examples) == 1:
        axes = [axes]

    for axis, row in zip(axes, examples.itertuples(index=False)):
        image_path = DATASET_ROOT / row.relative_image_path
        result = predictor.predict(image_path).to_dict()
        with Image.open(image_path) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(
            f"Truth={int(row.binary_label)} | Prediction={result['label']} | "
            f"AMD probability={result['probability']:.4f}"
        )
        axis.axis("off")
        print(json.dumps({"image_id": row.image_id, **result}, indent=2, ensure_ascii=False))

    plt.tight_layout()
    plt.show()
else:
    print("Standalone smoke test is disabled.")

## 7. Save the frozen model package

Use **Save Version** or **Quick Save with Output enabled**. Confirm this directory appears in the Notebook output:

```text
/kaggle/working/eyeai_model_package/run09_tta_v1
```

Then create a Kaggle Dataset from that directory. The next project patch will use this dataset as the only model input for FastAPI; training checkpoints and the official RETFound base checkpoint will no longer be required.